# AOMIC Data

---


### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Data access via SSHFS

---


In [60]:
# Load the participant.tsv files
participant_dfs = {}

key = "ID1000"
participant_dfs[key] = pd.read_csv("/home/ubuntu/cambridgemount/OpenNeuro_ID1000/participants.tsv", sep="\t")[['participant_id','age','sex']].dropna()
participant_dfs[key]['unique_id'] = f"{key}_" + participant_dfs[key]['participant_id'].astype(str)
participant_dfs[key]['site'] = key
participant_dfs[key]['sex'] = participant_dfs[key]['sex'].map({'female':'F', 'male': 'M'})
participant_dfs[key]['path'] = (
    "/home/ubuntu/cambridgemount/OpenNeuro_ID1000/derivatives/" +
    participant_dfs[key]['participant_id'].astype(str) +
    "/run-1/surfaces/" + participant_dfs[key]['participant_id'].astype(str)
)

key = "PIOP1"
# participant_dfs[key] = pd.read_csv("/home/ubuntu/cambridgemount/OpenNeuro_PIOP1/participants.tsv", sep="\t")[['participant_id','age','sex']]
participant_dfs[key] = pd.read_csv(
    "https://s3.amazonaws.com/openneuro.org/ds002785/participants.tsv?versionId=92d2he89mk1LAcDnfD5Q4LIVK1gmZqEv",
    sep="\t")[['participant_id','age','sex']].dropna()
participant_dfs[key]['unique_id'] = f"{key}_" + participant_dfs[key]['participant_id'].astype(str)
participant_dfs[key]['site'] = key
participant_dfs[key]['sex'] = participant_dfs[key]['sex'].map({'F':'F', 'M': 'M'})
participant_dfs[key]['path'] = (
    "/home/ubuntu/cambridgemount/OpenNeuro_PIOP1/derivatives/freesurfer/" +
    participant_dfs[key]['participant_id'].astype(str) +
    "/run-1/surfaces/" + participant_dfs[key]['participant_id'].astype(str)
)

key = "PIOP2"
participant_dfs[key] = pd.read_csv("/home/ubuntu/cambridgemount/OpenNeuro_PIOP2/participants.tsv", sep="\t")[['participant_id','age','sex']].dropna()
participant_dfs[key]['unique_id'] = f"{key}_" + participant_dfs[key]['participant_id'].astype(str)
participant_dfs[key]['site'] = key
participant_dfs[key]['sex'] = participant_dfs[key]['sex'].map({'F':'F', 'M': 'M'})
participant_dfs[key]['path'] = (
    "/home/ubuntu/cambridgemount/OpenNeuro_PIOP2/derivatives//freesurfer/" +
    participant_dfs[key]['participant_id'].astype(str) +
    "/run-1/surfaces/" + participant_dfs[key]['participant_id'].astype(str)
)

participant_df = pd.concat(
    [participant_dfs[x] for x in participant_dfs],
    ignore_index=True
)

participant_df.shape

(1361, 6)

In [ ]:
participant_df.head()

In [47]:
participant_df['sex'].value_counts(dropna=False)

sex
F    731
M    630
Name: count, dtype: int64

In [48]:
participant_df['site'].value_counts(dropna=False)

site
ID1000    928
PIOP2     224
PIOP1     209
Name: count, dtype: int64

## Extracting data

---

In [69]:
dataset = "AOMIC"

subjects = list(participant_df["unique_id"])
paths = list(participant_df["path"])

len(subjects)


1361

In [70]:
import joblib
import tarfile, os, shutil
import numpy as np

items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def process_subject(idx, subject, freesurfer_path, dataset):
    sub_dir = f"{idx:02d}"[-2:]
    
    sshfs_directory = freesurfer_path
    freesurfer_directory = f"/mountpoint/data/CSD3/snm_thickness/{dataset}/freesurfer/{subject}/"
    thickness_fslr_output = f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"
    euler_number_output = f"/mountpoint/data/normative/eno/{dataset}/{sub_dir}/{subject}.eno.npy"

    if os.path.exists(thickness_fslr_output):
        return f"Skipping {subject}, already processed"

    if not Path(sshfs_directory).exists():
        return f"Skipping {subject}, no sshfs directory at {sshfs_directory}"

    # Copy needed files
    for item in items:
        src_file = Path(sshfs_directory) / f"surf/{item}"
        dst_file = Path(freesurfer_directory) / f"surf/{item}"
        # Make sure source exists
        if src_file.exists():
            # Make sure destination exists
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            # Copy the file
            shutil.copy(src_file, dst_file)
        else:
            return f"Skipping {subject}, missing file: {src_file}"

    # Compute
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
    np.save(
        ensure_dir(thickness_fslr_output),
        transformed_fslr_thickness.astype(np.float32)
    )
    euler_number = snm.utils.nitools.compute_total_euler_number(freesurfer_directory)
    np.save(
        ensure_dir(euler_number_output),
        euler_number
    )

    # Cleanup
    shutil.rmtree(freesurfer_directory, ignore_errors=True)

    return f"Done {subject}"


In [ ]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=56, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject)(idx, subject, paths[idx], dataset)
    for idx, subject in enumerate(subjects)
)


In [ ]:
results

In [ ]:
# list of valid subjects
valid_subjects = [
    subject
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict = {
    subject: {
        "unique_id": subject,
        "participant_id": subject,
        "session_id": 1,
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'F': 'F',
    'M': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_df.iterrows()):
    key = row["unique_id"]
    if key in valid_subjects_dict:
        # if row['sex'] in sex_encoder:
        valid_subjects_dict[key]['age'] = row['age']
        valid_subjects_dict[key]['sex'] = sex_encoder[row['sex']]
        valid_subjects_dict[key]['site'] = row['site']
        # valid_subjects_dict[key]['diagnosis'] = (row['dx'] != 'CN')
        # else:
        #     valid_subjects_dict.pop(key)

# mean thickness, euler number and validity check
for key in tqdm(valid_subjects_dict):
    idx = valid_subjects_dict[key]["subject_index"]
    subject = valid_subjects_dict[key]["unique_id"]
    valid_subjects_dict[key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[key]["euler_no"] = int(np.load(
        f"/mountpoint/data/normative/eno/{dataset}/{(idx%100):02d}/{subject}.eno.npy",
    ))
    valid_subjects_dict[key]["validity_check"] = (
        # (valid_subjects_dict[key]["diagnosis"] == False)  # Exclude those with a diagnosis
        # and
        (valid_subjects_dict[key]["thickness"] != np.nan)  # Exclude those missing thickness data
        and
        (valid_subjects_dict[key]["euler_no"] != np.nan)  # Exclude those missing euler number
    )

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [85]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects)
)


In [86]:
import joblib

joblib.dump(valid_subjects_dict, ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/AOMIC/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [91]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(1358, 9)

## Scratch

---